In [7]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
df = pd.read_csv("../data/processed/amazon_cleaned.csv")

df.head()

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,img_link,product_link,combined_features
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers Accessories Accessories Peripherals ...,399.0,1099.0,64.0,4.2,24269.0,High Compatibility : Compatible With iPhone 12...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...,Wayona Nylon Braided USB to Lightning Fast Cha...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers Accessories Accessories Peripherals ...,199.0,349.0,43.0,4.0,43994.0,"Compatible with all Type C enabled devices, be...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers Accessories Accessories Peripherals ...,199.0,1899.0,90.0,3.9,7928.0,【 Fast Charger& Data Sync】-With built-in safet...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...,Sounce Fast Phone Charging Cable & Data Sync U...
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers Accessories Accessories Peripherals ...,329.0,699.0,53.0,4.2,94363.0,The boAt Deuce USB 300 2 in 1 cable is compati...,https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers Accessories Accessories Peripherals ...,154.0,399.0,61.0,4.2,16905.0,[CHARGE & SYNC FUNCTION]- This cable comes wit...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...,Portronics Konnect L 1.2M Fast Charging 3A 8 P...


In [9]:
print(df.shape)

(1337, 12)


In [10]:
df.isnull().sum()

product_id             0
product_name           0
category               0
discounted_price       0
actual_price           0
discount_percentage    0
rating                 0
rating_count           0
about_product          0
img_link               0
product_link           0
combined_features      0
dtype: int64

In [11]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(df["combined_features"])

print(tfidf_matrix.shape)

(1337, 5000)


In [12]:
cosine_sim = cosine_similarity(tfidf_matrix)

print(cosine_sim.shape)

(1337, 1337)


In [13]:
indices = {}

for idx, product in enumerate(df["product_name"]):
    if product not in indices:
        indices[product] = idx

In [27]:
def recommend_products(product_name, top_n=10):

    if product_name not in indices:
        return "Product not found!"

    idx = indices[product_name]

    similarity_scores = list(enumerate(cosine_sim[idx]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n+1]

    recommendations = []

    for product_index, similarity in similarity_scores:
        recommendations.append({
            "product_name": df.iloc[product_index]["product_name"],
            "category": df.iloc[product_index]["category"],
            "discounted_price": df.iloc[product_index]["discounted_price"],
            "rating": df.iloc[product_index]["rating"],
            "similarity_score": round(float(similarity), 4)
        })

    return pd.DataFrame(recommendations)

In [28]:
df["product_name"].head(10)

0    Wayona Nylon Braided USB to Lightning Fast Cha...
1    Ambrane Unbreakable 60W / 3A Fast Charging 1.5...
2    Sounce Fast Phone Charging Cable & Data Sync U...
3    boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...
4    Portronics Konnect L 1.2M Fast Charging 3A 8 P...
5    pTron Solero TB301 3A Type-C Data and Fast Cha...
6    boAt Micro USB 55 Tangle-free, Sturdy Micro US...
7               MI Usb Type-C Cable Smartphone (Black)
8    TP-Link USB WiFi Adapter for PC(TL-WN725N), N1...
9    Ambrane Unbreakable 60W / 3A Fast Charging 1.5...
Name: product_name, dtype: str

In [29]:
product = df.loc[0, "product_name"]

print(product)

print(indices[product])

print(type(indices[product]))

Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey)
0
<class 'int'>


In [17]:
type(indices[df.loc[0, "product_name"]])

int

In [18]:
print(type(indices[df.loc[0, "product_name"]]))

<class 'int'>


In [19]:
print(type(cosine_sim))
print(cosine_sim.shape)

print(type(cosine_sim[0]))
print(type(cosine_sim[0][0]))

<class 'numpy.ndarray'>
(1337, 1337)
<class 'numpy.ndarray'>
<class 'numpy.float64'>


In [20]:
duplicates = df["product_name"].duplicated().sum()

print("Duplicate product names:", duplicates)

Duplicate product names: 0


In [21]:
idx = indices[df.loc[0, "product_name"]]

print(idx)
print(type(idx))

similarity_scores = list(enumerate(cosine_sim[idx]))

print(type(similarity_scores[0][1]))
print(similarity_scores[0][1])

0
<class 'int'>
<class 'numpy.float64'>
0.9999999999999998


In [22]:
idx = 0

similarity_scores = list(enumerate(cosine_sim[idx]))

print(similarity_scores[:5])

sorted_scores = sorted(
    similarity_scores,
    key=lambda x: x[1],
    reverse=True
)

print(sorted_scores[:5])

[(0, np.float64(0.9999999999999998)), (1, np.float64(0.15958836628013257)), (2, np.float64(0.3486748253572278)), (3, np.float64(0.14137688453472527)), (4, np.float64(0.42651206173562145))]
[(0, np.float64(0.9999999999999998)), (220, np.float64(0.9488196142261819)), (42, np.float64(0.917867811105573)), (89, np.float64(0.9165309454459419)), (80, np.float64(0.7738257070839918))]


In [30]:
recommend_products(df.loc[0, "product_name"])

,product_name,category,discounted_price,rating,similarity_score
0,Wayona Nylon Braided Usb Syncing And Charging ...,Computers Accessories Accessories Peripherals ...,649.0,4.2,0.9488
1,Wayona Nylon Braided 3A Lightning to USB A Syn...,Computers Accessories Accessories Peripherals ...,399.0,4.2,0.9179
2,Wayona Nylon Braided (2 Pack) Lightning Fast U...,Computers Accessories Accessories Peripherals ...,649.0,4.2,0.9165
3,Wayona Usb Nylon Braided Data Sync And Chargin...,Computers Accessories Accessories Peripherals ...,399.0,4.2,0.7738
4,Wayona Nylon Braided Lightning USB Data Sync &...,Computers Accessories Accessories Peripherals ...,399.0,4.2,0.7488
5,Wayona Nylon Braided 2M / 6Ft Fast Charge Usb ...,Computers Accessories Accessories Peripherals ...,449.0,4.2,0.7400
6,MYVN LTG to USB for Fast Charging & Data Sync ...,Computers Accessories Accessories Peripherals ...,252.0,3.7,0.6401
7,Wayona Nylon Braided USB Data Sync and Fast Ch...,Computers Accessories Accessories Peripherals ...,349.0,4.2,0.5160
8,SWAPKART Fast Charging Cable and Data Sync USB...,Computers Accessories Accessories Peripherals ...,209.0,3.9,0.5137
9,"REDTECH USB-C to Lightning Cable 3.3FT, [Apple...",Computers Accessories Accessories Peripherals ...,249.0,5.0,0.4952


In [24]:
df.loc[[0, 369, 614]]

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,img_link,product_link,combined_features
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers Accessories Accessories Peripherals ...,399.0,1099.0,64.0,4.2,24269.0,High Compatibility : Compatible With iPhone 12...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...,Wayona Nylon Braided USB to Lightning Fast Cha...
369,B0B3N7LR6K,"Fire-Boltt Visionary 1.78"" AMOLED Bluetooth Ca...",Electronics WearableTechnology SmartWatches,3999.0,16999.0,76.0,4.3,17159.0,Fire-Boltt is India' No 1 Wearable Watch Brand...,https://m.media-amazon.com/images/I/41r1d8a2WG...,https://www.amazon.in/Fire-Boltt-Smartwatch-Re...,"Fire-Boltt Visionary 1.78"" AMOLED Bluetooth Ca..."
614,B01M72LILF,"Logitech M221 Wireless Mouse, Silent Buttons, ...",Computers Accessories Accessories Peripherals ...,799.0,1295.0,38.0,4.4,34852.0,Enjoy the sound of silence. With the same clic...,https://m.media-amazon.com/images/I/31ROHZJMEU...,https://www.amazon.in/Logitech-Silent-Wireless...,"Logitech M221 Wireless Mouse, Silent Buttons, ..."


In [25]:
print("Duplicate Product IDs:", df["product_id"].duplicated().sum())
print("Duplicate Product Names:", df["product_name"].duplicated().sum())

Duplicate Product IDs: 0
Duplicate Product Names: 0


In [26]:
print(df.shape)

(1337, 12)
